In [ ]:
import matplotlib.pyplot as plt 
import json 
import numpy as np 
import pandas as pd
import os 

jsons_Ni_Mo = "Datasets"
stability_dicts = {}
EIS_dicts = {}
# Loop over all the datasets with the stability data
for json_file in os.listdir(jsons_Ni_Mo):
    if "Stability" in json_file:
        file_path = os.path.join(jsons_Ni_Mo, json_file)
        
        with open(file_path, 'r') as infile:
            stability_dict = json.load(infile)
        
        # Merge dictionaries
        stability_dicts.update(stability_dict)

# Loop over all the datasets with EIS data
for json_file in os.listdir(jsons_Ni_Mo):
    if "EIS" in json_file:
        file_path = os.path.join(jsons_Ni_Mo, json_file)
        
        with open(file_path, 'r') as infile:
            EIS_dict = json.load(infile)
        
        # Merge dictionaries
        EIS_dicts.update(EIS_dict)




# Create df from json files

In [ ]:

rows = []
# ## NiMo
for experiment, exp_data in stability_dicts.items():
    params = exp_data['ML optimization params']
    timestamp = exp_data["timestamp"]
    R_100 = exp_data['Cycling results']['Resistance at 100 mA/cm2']

    overpotentials_1 = exp_data['Cycling results']["Overpotentials at 1 mA/cm2"]
    overpotentials_10 = exp_data['Cycling results']["Overpotentials at 10 mA/cm2"]
    overpotentials_100 = exp_data['Cycling results']["Overpotentials at 100 mA/cm2"]

    measured_current_density_100 = exp_data['Cycling results']["Measured current density at 100 mA/cm2"]
    
    overpotentials_mean_1 = np.mean(overpotentials_1)
    overpotentials_mean_10 = np.mean(overpotentials_10)
    overpotentials_mean_100 = np.mean(overpotentials_100)
    current_density_100_mean = np.mean(measured_current_density_100)
    # Convert params to numeric where possible
    params_numeric = {
        k: float(v) if v is not None else np.nan
        for k, v in params.items()
    }
    
    row = {
        'experiment': experiment,
        'timestamp' : timestamp, 
        
        'OP @ -1 mA/cm2 mean (mV)' : overpotentials_mean_1,
        'OP @ -10 mA/cm2 mean (mV)' : overpotentials_mean_10,
        'OP @ -100 mA/cm2 mean (mV)' : overpotentials_mean_100,
        'Measured I (mA/cm2) @ 100' : current_density_100_mean,
        **params_numeric
    }
    rows.append(row)


df_NiMo = pd.DataFrame(rows)
df_NiMo = df_NiMo.sort_values(by='OP @ -10 mA/cm2 mean (mV)')


# Sort by OP at -100 mA/cm2
df_NiMo = df_NiMo.sort_values(by='OP @ -100 mA/cm2 mean (mV)')

# Reset index
df_NiMo = df_NiMo.reset_index(drop=True)

df_NiMo['time_idx'] = df_NiMo.index


# Investigate degradation during cycling of the best catalysts at -1, -10 and -100 mA/cm2